# Player features and model evaluation

## tl;dr

- 47,601 pinned scorer events produced 27 prior-only player-threat features.
- The optional player + LightGBM blend reached rolling Log Loss **0.8662**.
- Final holdout Log Loss was **0.8169**, versus **0.8153** for the simpler core champion.
- The candidate was correctly **not promoted**.
- Full squad, injury, availability, lineup, expected-minutes, and player-rating
  features are implemented, but require timestamped historical snapshots before
  they can enter training.

## Context & Methods

All model choices use expanding chronological folds for 2018, 2021, 2022,
2023, and 2024. No random split is used. Player information must be available
strictly before kickoff. Scorer-event coverage is modeled explicitly because
the source is incomplete.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

coverage = json.loads((ROOT / "reports/player_features/coverage.json").read_text())
selection = json.loads((ROOT / "reports/player_model_search/selection.json").read_text())
validation = pd.read_csv(ROOT / "reports/player_model_search/rolling_aggregate.csv")
holdout = pd.read_csv(ROOT / "reports/player_model_search/holdout_results.csv")
champion_validation = pd.read_csv(ROOT / "reports/champion_model/validation_results.csv")
champion_holdout = pd.read_csv(ROOT / "reports/champion_model/holdout_results.csv")

## Data

In [2]:
pd.DataFrame({
    "measure": [
        "model rows",
        "scorer-derived features",
        "rows with both team histories",
        "team1 mean source coverage",
        "team2 mean source coverage",
    ],
    "value": [
        coverage["rows"],
        coverage["feature_count"],
        coverage["rows_with_both_scorer_histories"],
        coverage["team1_mean_source_coverage"],
        coverage["team2_mean_source_coverage"],
    ],
})

,measure,value
0,model rows,32252.000000
1,scorer-derived features,27.000000
2,rows with both team histories,31661.000000
3,team1 mean source coverage,0.426856
4,team2 mean source coverage,0.428093


The roughly 42.7% event completeness is a material limitation.
The model receives coverage and history-count columns so incomplete source data
is not silently interpreted as weak players.

## Results

In [3]:
validation[[
    "model", "log_loss", "brier", "rps", "accuracy", "calibration_error"
]].sort_values("log_loss")

,model,log_loss,brier,rps,accuracy,calibration_error
0,blend_player_scorer__lightgbm_player_scorer,0.866168,0.509023,0.168356,0.605437,0.010964
1,blend_player_scorer__hist_gbm_player_scorer,0.867259,0.509658,0.168545,0.604493,0.011466
2,logistic_fifa_baseline,0.869163,0.510720,0.169030,0.608458,0.017998
3,logistic_player_scorer,0.869268,0.510951,0.169078,0.604304,0.016143
4,lightgbm_player_scorer,0.875047,0.514286,0.170836,0.598452,0.015109
5,hist_gbm_player_scorer,0.876287,0.514874,0.170820,0.597697,0.012415


In [4]:
comparison = pd.concat([
    champion_holdout.assign(source="core champion"),
    holdout.assign(source="player candidate"),
], ignore_index=True)
comparison[[
    "source", "model", "log_loss", "brier", "rps", "accuracy",
    "calibration_error"
]].sort_values("log_loss")

,source,model,log_loss,brier,rps,accuracy,calibration_error
0,core champion,logistic_fifa_hist_gbm_blend,0.815270,0.478736,0.156308,0.619266,0.025374
2,player candidate,blend_player_scorer__lightgbm_player_scorer,0.816907,0.479799,0.156781,0.620031,0.023587
1,core champion,compact_no_fifa_baseline,0.832533,0.488663,0.160892,0.613150,0.027864


## Snapshot feature contract

In [5]:
template = pd.read_csv(ROOT / "data/templates/player_snapshots.csv")
pd.DataFrame({"supported_column": template.columns})

,supported_column
0,snapshot_id
1,player_id
2,player_name
3,team
4,available_at
5,snapshot_at
6,source
7,source_version
8,position
9,club


## Takeaways

1. Player-derived information can improve historical validation, but the
   improvement was too small and did not survive the final holdout.
2. The core champion remains the production model because it is simpler and
   has better holdout Log Loss, Brier, and RPS.
3. Official squad and lineup snapshots can now be ingested without leakage.
   They must not be promoted until matching historical snapshots cover the
   evaluation period.
4. `models/world_cup_2026_player_scorer.joblib` is an optional research artifact,
   not the default production model.